In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input

In [2]:
df = pd.read_csv('../data/TSLA.csv')

df['Date'] = pd.to_datetime(df['Date'])
df.set_index('Date', inplace=True)

data = df[['Close']]

In [3]:
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(data)

train_size = int(len(scaled_data) * 0.8)

train_data = scaled_data[:train_size]
test_data = scaled_data[train_size-60:]

In [4]:
def create_sequences(data, seq_length=60):
    X, Y = [], []

    for i in range(seq_length, len(data)):
        X.append(data[i-seq_length:i])
        Y.append(data[i])

    return np.array(X), np.array(Y)


X_train, Y_train = create_sequences(train_data)
X_test, Y_test = create_sequences(test_data)

In [5]:
model = Sequential()

model.add(Input(shape=(X_train.shape[1], 1)))
model.add(LSTM(50, return_sequences=False))
model.add(Dense(1))

model.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm (LSTM)                 (None, 50)                10400     
                                                                 
 dense (Dense)               (None, 1)                 51        
                                                                 
Total params: 10451 (40.82 KB)
Trainable params: 10451 (40.82 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [6]:
model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

In [7]:
history = model.fit(
    X_train, Y_train,
    epochs=10,
    batch_size=32
)

Epoch 1/10

59/59 [==============================] - 2s 11ms/step - loss: 0.0031
Epoch 2/10
59/59 [==============================] - 1s 11ms/step - loss: 1.9164e-04
Epoch 3/10
59/59 [==============================] - 1s 11ms/step - loss: 1.8407e-04
Epoch 4/10
59/59 [==============================] - 1s 11ms/step - loss: 1.8122e-04
Epoch 5/10
59/59 [==============================] - 1s 12ms/step - loss: 1.8582e-04
Epoch 6/10
59/59 [==============================] - 1s 14ms/step - loss: 1.6676e-04
Epoch 7/10
59/59 [==============================] - 1s 11ms/step - loss: 1.6080e-04
Epoch 8/10
59/59 [==============================] - 1s 11ms/step - loss: 1.5290e-04
Epoch 9/10
59/59 [==============================] - 1s 11ms/step - loss: 1.4490e-04
Epoch 10/10
59/59 [==============================] - 1s 11ms/step - loss: 1.4423e-04


In [10]:
model.save("../models/lstm_fixed.h5",save_format="h5")

c:\Users\Zbook\Desktop\TSLA-stock-prediction\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
